# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:

%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)

clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [5]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


In [6]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


In [7]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))

base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.545     0.335     0.415      9389
           1      0.684     0.838     0.753     16162

    accuracy                          0.653     25551
   macro avg      0.615     0.586     0.584     25551
weighted avg      0.633     0.653     0.629     25551



In [8]:
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                        AND f.report_date >  b.end_d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_mid30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 150          -- your custom threshold (was 100 in the 60-day version)
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_90d):,} content items with enough history (90-day window)')
features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

95,895 content items with enough history (90-day window)


,client_hash_id,content_hash_id,imp_last30,imp_mid30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,616.0,444.0,1.0,12.267505
1,client_e547b89c05043229,content_bab118937886d46a,110.0,173.0,186.0,0.0,20.211330
2,client_e547b89c05043229,content_5d6131702b65bee6,63.0,122.0,172.0,1.0,13.623843
3,client_e547b89c05043229,content_a64be0ba11772089,248.0,409.0,448.0,0.0,18.283926
4,client_e547b89c05043229,content_74369d7d3369d86b,192.0,358.0,778.0,2.0,20.736734


In [9]:
pos_volatility = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    )
    SELECT f.content_hash_id,
           STDDEV_POP(f.gsc_avg_position) AS pos_volatility_last30
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 30 DAY
    GROUP BY 1
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data_90d = (features_90d
            .merge(qsignals, on='content_hash_id', how='left')
            .merge(pos_volatility, on='content_hash_id', how='left'))

print(f'joined: {len(data_90d):,} rows')
data_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 95,895 rows


,client_hash_id,content_hash_id,imp_last30,imp_mid30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,pos_volatility_last30
0,client_e547b89c05043229,content_25dfa3e39bc37247,216.0,616.0,444.0,1.0,12.267505,5.0,0.038401,0.897335,34.0,82.0,0.414634,15.085233
1,client_e547b89c05043229,content_bab118937886d46a,110.0,173.0,186.0,0.0,20.211330,1.0,0.168443,0.528785,142.0,142.0,1.000000,12.827683
2,client_e547b89c05043229,content_5d6131702b65bee6,63.0,122.0,172.0,1.0,13.623843,2.0,0.162465,0.767507,14.0,25.0,0.560000,15.417691
3,client_e547b89c05043229,content_a64be0ba11772089,248.0,409.0,448.0,0.0,18.283926,1.0,0.081448,0.902262,18.0,18.0,1.000000,12.150734
4,client_e547b89c05043229,content_74369d7d3369d86b,192.0,358.0,778.0,2.0,20.736734,9.0,0.054970,0.509036,401.0,579.0,0.692573,14.556961


In [10]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

data_90d['is_declining'] = (data_90d['imp_last30'] < 0.8 * data_90d['imp_mid30']).astype(int)

feature_cols_90d = ['imp_mid30', 'visible_queries', 'rare_share', 'anon_share',
                     'top_query_share', 'pos_volatility_last30']
model_data_90d = data_90d.dropna(subset=feature_cols_90d)

X = model_data_90d[feature_cols_90d]
y = model_data_90d['is_declining']
groups = model_data_90d['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

# sanity check: no client should appear in both splits
assert set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]) == set()

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'clients in train: {groups.iloc[train_idx].nunique()}, clients in test: {groups.iloc[test_idx].nunique()}')
print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model_grouped.predict(X_te), digits=3))


clients in train: 31, clients in test: 11
base rate (always predict majority): 0.701
              precision    recall  f1-score   support

           0      0.604     0.661     0.631      2693
           1      0.850     0.816     0.832      6320

    accuracy                          0.769      9013
   macro avg      0.727     0.738     0.732      9013
weighted avg      0.776     0.769     0.772      9013



## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [11]:
'''I chose Random Forest for this task because the goal is to identify pages that are likely to need content refresh. The relationship between search performance signals and declining performance may not be linear, and Random Forest can capture interactions between multiple features.

The model will use search-performance features such as previous impressions, query diversity, top-query share, and ranking volatility. The output will help rank pages for content review.

I am not choosing the model because it is more complex. It is useful only if it performs better than the Week-4 hand-written baseline on the same validation data and metric.'''

'I chose Random Forest for this task because the goal is to identify pages that are likely to need content refresh. The relationship between search performance signals and declining performance may not be linear, and Random Forest can capture interactions between multiple features.\n\nThe model will use search-performance features such as previous impressions, query diversity, top-query share, and ranking volatility. The output will help rank pages for content review.\n\nI am not choosing the model because it is more complex. It is useful only if it performs better than the Week-4 hand-written baseline on the same validation data and metric.'

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [12]:
'''Client A ──→ TRAIN

Client B ──→ TRAIN

Client C ──→ TEST'''

'Client A ──→ TRAIN\n\nClient B ──→ TRAIN\n\nClient C ──→ TEST'

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.